In [35]:
import numpy as np
import gymnasium as gym

# Create environment
env = gym.make("CartPole-v1")

# Discretization helper
class Discretizer:
    def __init__(self):
        self.bins = [
            np.linspace(-4.8, 4.8, 10),     # cart position
            np.linspace(-5, 5, 10),         # cart velocity
            np.linspace(-0.418, 0.418, 10), # pole angle
            np.linspace(-5, 5, 10)          # pole velocity
        ]

    def discretize(self, state):
        return tuple(
            np.digitize(s, b) for s, b in zip(state, self.bins)
        )

# Q-Learning Agent
class QLearningAgent:
    def __init__(self, env, learning_rate=0.1, discount_factor=0.99,
                 epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01):

        self.env = env
        self.action_space = env.action_space.n
        self.discretizer = Discretizer()

        # 10 bins per state dimension → 10^4 states
        self.q_table = np.zeros((10, 10, 10, 10, self.action_space))

        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

    def choose_action(self, state):
        state = self.discretizer.discretize(state)

        if np.random.random() < self.epsilon:
            return self.env.action_space.sample()
        else:
            return np.argmax(self.q_table[state])

    def update_q_table(self, state, action, reward, next_state):
        state = self.discretizer.discretize(state)
        next_state = self.discretizer.discretize(next_state)

        best_next_action = np.argmax(self.q_table[next_state])

        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]

        self.q_table[state][action] += self.lr * td_error

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)


# Training loop
agent = QLearningAgent(env)
episodes = 1000

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        agent.update_q_table(state, action, reward, next_state)

        state = next_state
        total_reward += reward

    agent.decay_epsilon()

    if episode % 100 == 0:
        print(f"Episode {episode}, Reward: {total_reward}, Epsilon: {agent.epsilon:.3f}")


# Test the trained agent
state, _ = env.reset()
done = False

while not done:
    action = np.argmax(agent.q_table[agent.discretizer.discretize(state)])
    state, _, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

env.close()

Episode 0, Reward: 22.0, Epsilon: 0.995
Episode 100, Reward: 26.0, Epsilon: 0.603
Episode 200, Reward: 19.0, Epsilon: 0.365
Episode 300, Reward: 17.0, Epsilon: 0.221
Episode 400, Reward: 21.0, Epsilon: 0.134
Episode 500, Reward: 17.0, Epsilon: 0.081
Episode 600, Reward: 17.0, Epsilon: 0.049
Episode 700, Reward: 21.0, Epsilon: 0.030
Episode 800, Reward: 18.0, Epsilon: 0.018
Episode 900, Reward: 21.0, Epsilon: 0.011
